# 💳 Credit Card Fraud Detection using Logistic Regression

**Author:** Rohann Neupane 
**Dataset:** [Credit Card Fraud Detection — Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Model:** Logistic Regression  

This notebook builds a machine learning model to detect fraudulent credit card transactions. Because fraud cases are extremely rare compared to legitimate ones, we use **undersampling** to balance the dataset before training.

---

## 0. ⚙️ Setup — Download Dataset from Kaggle

**Only needed when running on Google Colab or a fresh machine.**  
Skip this section if you already have `creditcard.csv` in your working directory.

### How to get your `kaggle.json`:
1. Go to **https://www.kaggle.com**
2. Click your profile picture → **Settings**
3. Scroll to **API** section → Click **Create New Token**
4. A file called `kaggle.json` will download to your computer
5. Upload it in the cell below when prompted

In [ ]:
# -------------------------------------------------------
# RUN THIS CELL ONLY ON GOOGLE COLAB
# Skip if running locally and creditcard.csv is already present
# -------------------------------------------------------

import os

if not os.path.exists('creditcard.csv'):
    # Step 1: Upload your kaggle.json
    from google.colab import files
    print('Upload your kaggle.json file:')
    files.upload()

    # Step 2: Move it to the correct location
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

    # Step 3: Download dataset
    !pip install kaggle -q
    !kaggle datasets download -d mlg-ulb/creditcardfraud
    !unzip -q creditcardfraud.zip
    print('✅ Dataset downloaded successfully!')
else:
    print('✅ creditcard.csv already exists, skipping download.')

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 2. Load Dataset

In [ ]:
#Loading data to pandas dataframe
credit_card_data= pd.read_csv('creditcard.csv')

#rows of the dataset
credit_card_data #dataset information
credit_card_data.info()
#checking the number of missing values in each columns
credit_card_data.isnull().sum() #distribution of legit transcations and fraudelent transaction
credit_card_data['Class'].value_counts()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
 #seperating the data for analysis
legit= credit_card_data[credit_card_data.Class==0]
fraud= credit_card_data[credit_card_data.Class==1]  #statistical measures of the data
legit.Amount.describe()
 

print(legit.shape)
print(fraud.shape)

In [ ]:
fraud.Amount.describe()

In [ ]:
#compairing the values for both transaction
credit_card_data.groupby('Class').mean()

## 4. Balancing the Dataset (Undersampling)

The dataset is highly imbalanced — only 492 fraud cases out of 284,807 transactions. We undersample the legitimate transactions to match the number of fraud cases so the model doesn't just learn to predict "legit" every time.

In [ ]:
legit_sample = legit.sample(n=492)

In [ ]:
new_dataset= pd.concat([legit_sample, fraud], axis= 0)
print(new_dataset)

In [ ]:
new_dataset['Class'].value_counts()

In [ ]:
new_dataset.groupby('Class').mean()

## 5. Splitting Features and Target

In [ ]:
X = new_dataset.drop(columns='Class', axis=1)
Y = new_dataset['Class']

print(X, Y)

## 6. Model Training

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)
print(X.shape, X_train.shape,X_test.shape) 
model = LogisticRegression()
#Training the logistic regression model data
model.fit(X_train, Y_train)

## 7. Model Evaluation

In [ ]:
#Accuracy score on training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)
print('Accuracy on training data:', training_data_accuracy)

#Accuracy on test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)
print('Accuracy on test data:', test_data_accuracy)

## 8. Predict on a New Transaction

Enter the feature values of a transaction below to check if it is **Fraudulent** or **Legitimate**.

In [ ]:
# -------------------------------------------------------
# USER INPUT — paste a row from the dataset to test
# V1 to V28 are PCA-transformed features (anonymized)
# Amount = transaction amount in dollars
# Time = seconds elapsed since first transaction in dataset
# -------------------------------------------------------

input_data = {
    'Time':   0,
    'V1':    -1.3598071,  'V2':  -0.0727811,  'V3':   2.5363467,
    'V4':     1.3781552,  'V5':  -0.3383208,  'V6':   0.4623878,
    'V7':     0.2395986,  'V8':   0.0986979,  'V9':   0.3637870,
    'V10':    0.0907941,  'V11':  -0.5515995, 'V12':  -0.6178009,
    'V13':   -0.9913898,  'V14': -0.3111694,  'V15':   1.4681770,
    'V16':   -0.4704005,  'V17':  0.2079708,  'V18':   0.0257905,
    'V19':    0.4039936,  'V20':  0.2514121,  'V21':  -0.0183068,
    'V22':    0.2778376,  'V23': -0.1104740,  'V24':   0.0669281,
    'V25':    0.1285394,  'V26': -0.1891148,  'V27':   0.1335584,
    'V28':   -0.0210530,
    'Amount': 149.62
}

input_df = pd.DataFrame([input_data])
prediction = model.predict(input_df)[0]

print("=" * 40)
if prediction == 0:
    print("  ✅ Transaction is LEGITIMATE")
else:
    print("  🚨 Transaction is FRAUDULENT")
print("=" * 40)